In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq


# ============================================================
# Model:
#
# f(x) = p phi(x + d/2) + (1-p) phi(x - d/2)
#
# Equal component variances:
# sigma_1 = sigma_2 = 1, so rho = 1.
#
# t = h / sigma = h
# n is fixed.
# ============================================================

SQRT_PI = np.sqrt(np.pi)
SQRT_2PI = np.sqrt(2.0 * np.pi)


def C_a(a: float, t, p: float, d: float):
    """
    Integral interaction term C_a.

    a = 0, 1, or 2.
    """
    t = np.asarray(t)

    A = p**2 + (1.0 - p)**2
    B = 2.0 * p * (1.0 - p)

    s = 2.0 + a * t**2

    return (
        A + B * np.exp(-d**2 / (2.0 * s))
    ) / np.sqrt(2.0 * np.pi * s)


def mise(t, p: float, d: float, n: int = 50):
    """
    Rescaled exact finite-sample MISE.
    """
    t = np.asarray(t)

    return (
        1.0 / (2.0 * SQRT_PI * n * t)
        + (1.0 - 1.0 / n) * C_a(2.0, t, p, d)
        - 2.0 * C_a(1.0, t, p, d)
        + C_a(0.0, t, p, d)
    )


def dC_dt(a: float, t, p: float, d: float):
    """
    Analytic derivative of C_a with respect to t.
    """
    t = np.asarray(t)

    if a == 0:
        return np.zeros_like(t, dtype=float)

    A = p**2 + (1.0 - p)**2
    B = 2.0 * p * (1.0 - p)

    D = d**2
    s = 2.0 + a * t**2
    E = np.exp(-D / (2.0 * s))

    return (
        a
        * t
        / SQRT_2PI
        * s ** (-2.5)
        * (-A * s + B * E * (D - s))
    )


def dmise_dt(t, p: float, d: float, n: int = 50):
    """
    Analytic derivative of the rescaled MISE.
    """
    return (
        -1.0 / (2.0 * SQRT_PI * n * t**2)
        + (1.0 - 1.0 / n) * dC_dt(2.0, t, p, d)
        - 2.0 * dC_dt(1.0, t, p, d)
    )


def second_derivative(t: float, p: float, d: float, n: int = 50):
    """
    Numerical derivative of the analytic first derivative.
    Used only to classify stationary points.
    """
    step = min(
        1e-4 * max(1.0, t),
        0.25 * t,
    )

    return (
        dmise_dt(t + step, p, d, n)
        - dmise_dt(t - step, p, d, n)
    ) / (2.0 * step)


# ============================================================
# Find all positive stationary points
# ============================================================

def stationary_points(
    p: float,
    d: float,
    n: int = 50,
    t_min: float = 1e-5,
    t_max: float | None = None,
    grid_size: int = 2500,
):
    """
    Isolate roots of d/dt MISE(t) using a logarithmic grid
    and refine each sign-changing interval with Brent's method.
    """

    if t_max is None:
        # Increase the search range as separation increases.
        t_max = max(30.0, 4.0 * d + 20.0)

    t_grid = np.geomspace(t_min, t_max, grid_size)
    derivative_values = dmise_dt(t_grid, p, d, n)

    roots = []

    for left, right, f_left, f_right in zip(
        t_grid[:-1],
        t_grid[1:],
        derivative_values[:-1],
        derivative_values[1:],
    ):
        if not (
            np.isfinite(f_left)
            and np.isfinite(f_right)
        ):
            continue

        if f_left * f_right < 0.0:
            root = brentq(
                lambda t: dmise_dt(t, p, d, n),
                left,
                right,
                xtol=1e-12,
                rtol=1e-12,
            )

            # Remove duplicate roots caused by adjacent brackets.
            if (
                not roots
                or abs(np.log(root / roots[-1])) > 1e-8
            ):
                roots.append(root)

    return roots


def local_minima(
    p: float,
    d: float,
    n: int = 50,
    **stationary_kwargs,
):
    """
    Return all stationary points with positive second derivative.
    """
    roots = stationary_points(
        p,
        d,
        n=n,
        **stationary_kwargs,
    )

    return [
        t
        for t in roots
        if second_derivative(t, p, d, n) > 0.0
    ]


# ============================================================
# Difference between the two minima
# ============================================================

def depth_gap(
    d: float,
    p: float,
    n: int = 50,
    **stationary_kwargs,
):
    """
    Gap = MISE(large-bandwidth minimum)
          - MISE(small-bandwidth minimum).

    A zero means equal-depth minima.
    """

    minima = local_minima(
        p,
        d,
        n=n,
        **stationary_kwargs,
    )

    if len(minima) < 2:
        raise ValueError(
            "The MISE has fewer than two local minima."
        )

    t1 = minima[0]
    t2 = minima[-1]

    return float(
        mise(t2, p, d, n)
        - mise(t1, p, d, n)
    )


# ============================================================
# Search the Maxwell curve in the (p,d)-plane
# ============================================================

def search_equal_minima(
    n: int = 50,
    p_values=None,
    d_values=None,
    gap_tolerance: float = 1e-8,
    grid_size: int = 2500,
):
    """
    For every fixed p, scan over d and detect sign changes
    of the difference between the two local-minimum values.
    """

    if p_values is None:
        # It is sufficient to search p <= 1/2 because
        # exchanging the components sends p to 1-p.
        p_values = np.linspace(0.02, 0.50, 97)

    if d_values is None:
        d_values = np.linspace(0.0, 30.0, 1201)

    solutions = []
    max_number_of_minima = 0

    for p in p_values:
        previous_valid_point = None

        for d in d_values:
            minima = local_minima(
                p,
                d,
                n=n,
                grid_size=grid_size,
            )

            max_number_of_minima = max(
                max_number_of_minima,
                len(minima),
            )

            if len(minima) < 2:
                # We have left the two-minimum region.
                previous_valid_point = None
                continue

            t1 = minima[0]
            t2 = minima[-1]

            gap = float(
                mise(t2, p, d, n)
                - mise(t1, p, d, n)
            )

            # Catch a grid point already extremely close
            # to an equal-depth pair.
            if abs(gap) < gap_tolerance:
                solutions.append(
                    (p, d, t1, t2, gap)
                )

            if previous_valid_point is not None:
                d_left, gap_left = previous_valid_point

                # The global minimizer changes identity.
                if gap_left * gap < 0.0:
                    try:
                        d_star = brentq(
                            lambda x: depth_gap(
                                x,
                                p,
                                n=n,
                                grid_size=grid_size,
                            ),
                            d_left,
                            d,
                            xtol=1e-10,
                            rtol=1e-10,
                        )

                        minima_star = local_minima(
                            p,
                            d_star,
                            n=n,
                            grid_size=grid_size,
                        )

                        t1_star = minima_star[0]
                        t2_star = minima_star[-1]

                        gap_star = float(
                            mise(t2_star, p, d_star, n)
                            - mise(t1_star, p, d_star, n)
                        )

                        solutions.append(
                            (
                                p,
                                d_star,
                                t1_star,
                                t2_star,
                                gap_star,
                            )
                        )

                    except ValueError:
                        # This can happen if a very narrow
                        # two-minimum region was not resolved.
                        pass

            previous_valid_point = (d, gap)

    result = pd.DataFrame(
        solutions,
        columns=[
            "p",
            "d",
            "t1",
            "t2",
            "MISE_gap",
        ],
    )

    if result.empty:
        print(
            "No equal local-minimum pairs were found "
            "in the selected parameter region."
        )
        print(
            f"Maximum number of local minima observed: "
            f"{max_number_of_minima}"
        )
        return result

    result = result.drop_duplicates(
        subset=["p", "d"]
    ).sort_values(["p", "d"])

    result["MISE"] = [
        mise(t1, p, d, n)
        for p, d, t1 in zip(
            result["p"],
            result["d"],
            result["t1"],
        )
    ]

    result["Rtt_t1"] = [
        second_derivative(t1, p, d, n)
        for p, d, t1 in zip(
            result["p"],
            result["d"],
            result["t1"],
        )
    ]

    result["Rtt_t2"] = [
        second_derivative(t2, p, d, n)
        for p, d, t2 in zip(
            result["p"],
            result["d"],
            result["t2"],
        )
    ]

    result = result.reset_index(drop=True)

    print(
        f"Maximum number of local minima observed: "
        f"{max_number_of_minima}"
    )
    print(
        f"Equal-minimum pairs found: {len(result)}"
    )

    return result

In [2]:
results = search_equal_minima(
    n=50,
    p_values=np.linspace(0.02, 0.50, 97),
    d_values=np.linspace(0.0, 30.0, 1201),
    grid_size=2500,
)

print(results)

No equal local-minimum pairs were found in the selected parameter region.
Maximum number of local minima observed: 1
Empty DataFrame
Columns: [p, d, t1, t2, MISE_gap]
Index: []
